In [1]:
import os
import matplotlib.pyplot as plt
import ansys.motorcad.core as pymotorcad
mcad = pymotorcad.MotorCAD(open_new_instance=False)
# 


In [ ]:
port=mcad.get_variable('MotorCADprocessID')


if port is None:
    mcad = pymotorcad.MotorCAD(open_new_instance=True)
else:
    print(port)


working_folder = os.getcwd()
   
if os.path.isdir(working_folder) is False:
    print("Working folder does not exist. Choose a folder that exists and try again.")
    print(working_folder)
    exit()

mcad.set_visible(visible=True)
parentDir=r'D:\KDHe10\e10_20251226'

In [ ]:
# Get all .mot files in parentDir
motFileList = []
for filename in os.listdir(parentDir):
	if filename.endswith('.mot'):
		full_path = os.path.join(parentDir, filename)
		motFileList.append({
			'filename': filename,
			'directory': parentDir,
			'full_path': full_path
		})

print(f"Found {len(motFileList)} .mot files:")
for i, mot_file in enumerate(motFileList):
	print(f"  [{i}] {mot_file['filename']}")

In [ ]:
mcad.load_from_file(mot_file['full_path'])

In [ ]:
mcadPath=mcad.get_variable("CurrentMotFilePath_MotorLAB")
mcadDir=mcad.get_variable("CurrentMotFileDir_MotorLAB")
mcad.set_variable("MessageDisplayState", 2)
mcad.show_magnetic_context()


# Export 2 Maxwell

In [41]:
mcad.set_variable('Ansys_ScriptFormat',0)
exportPyPath="E:\KDH\251114_C67_test\251114_C67_test.py"
mcad.export_to_ansys_electronics_desktop(file_path=exportPyPath)

<>:2: SyntaxWarning: invalid escape sequence '\K'
<>:2: SyntaxWarning: invalid escape sequence '\K'
C:\Users\user\AppData\Local\Temp\ipykernel_50032\2084949746.py:2: SyntaxWarning: invalid escape sequence '\K'
  exportPyPath="E:\KDH\251114_C67_test\251114_C67_test.py"


In [ ]:
exportPyPath

In [ ]:
import pathlib
from ansys.workbench.core import launch_workbench

## Fix Python Code

In [4]:
from pathlib import Path
ExportPyPath=r"D:\KDH"
ROOT = Path(ExportPyPath)
BAD_CHARS = ["�"]  # 필요하면 여기에 더 추가

def clean_text(text: str) -> str:
    for ch in BAD_CHARS:
        text = text.replace(ch, "")
    return text

def fix_file(path: Path) -> bool:
    try:
        # 일단 UTF-8로 읽고, 안 되면 대충이라도 읽어서 깨진 문자만 제거
        raw = path.read_bytes()
        text = raw.decode("utf-8", errors="replace")
    except Exception:
        return False

    new_text = clean_text(text)
    if new_text == text:
        return False  # 바뀐 게 없으면 스킵

    print(f"Fixing {path}")
    path.write_text(new_text, encoding="utf-8")
    return True

def main():
    fixed = 0
    for py in ROOT.rglob("*.py"):
        if fix_file(py):
            fixed += 1
    print(f"Done. Fixed {fixed} file(s).")

if __name__ == "__main__":
    main()

Fixing D:\KDH\e10_26R1mot.py
Fixing D:\KDH\새 폴더\EveryMotor\patch_infer.py
Fixing D:\KDH\새 폴더\EveryMotor\physicsnemo_train_from_pyMCAD.py
Fixing D:\KDH\새 폴더\EveryMotor\eMach\251114_C67_test_Moa_Max3D.py
Fixing D:\KDH\새 폴더\EveryMotor\eMach\Discovery.py
Fixing D:\KDH\새 폴더\EveryMotor\eMach\mlxperPJT\KETI\Discovery.py
Fixing D:\KDH\새 폴더\EveryMotor\eMach\tools\ansys\cbh.py
Done. Fixed 7 file(s).


# Make WB Material From Motor-CAD Mechanical Data

In [16]:
RotorMateriaName=mcad.get_variable('Material_Rotor_Lam_Back_Iron')

'20PN1150F_251114_C67_test'

In [17]:
MagnetMaterialName=mcad.get_variable('Material_Magnet')

In [30]:
# Motor-CAD 변수명(문자열) 리스트
mechanical_keys = [
    "YoungsCoefficient_RotorLam",
    "PoissonsRatio_RotorLam",
    "YieldStress_RotorLam",
    "Density_Rotor_Lam_Back_Iron",
]

# key -> value 딕셔너리로 한번에 가져오기
MechanicalDict = {k: mcad.get_variable(k) for k in mechanical_keys}

MechanicalDict

{'YoungsCoefficient_RotorLam': 192500,
 'PoissonsRatio_RotorLam': 0.3,
 'YieldStress_RotorLam': 370,
 'Density_Rotor_Lam_Back_Iron': 7650}

In [31]:
# (옵션) dict 값을 파이썬 변수처럼 쓰고 싶으면:
from types import SimpleNamespace

mech = SimpleNamespace(**MechanicalDict)

# 예:
# mech.YoungsCoefficient_RotorLam

# (옵션) 정말 "변수"로 만들고 싶다면(권장 X):
# globals().update(MechanicalDict)

In [ ]:
# 재료명(문자열)을 key로 쓰는 재료 물성 dict 만들기 (섹션: mechanical/thermal/magnetic)

# Motor-CAD에서 재료명 가져오기 (이미 위에서 실행했다면 재사용)
RotorMaterialName = mcad.get_variable('Material_Rotor_Lam_Back_Iron')

# Rotor lamination: Mechanical 섹션에 저장
lib.upsert_mechanical(RotorMaterialName, {
    "youngs_modulus": MechanicalDict.get("YoungsCoefficient_RotorLam"),
    "poissons_ratio": MechanicalDict.get("PoissonsRatio_RotorLam"),
    "yield_stress": MechanicalDict.get("YieldStress_RotorLam"),
    "density": MechanicalDict.get("Density_Rotor_Lam_Back_Iron"),
})

materials = lib.to_nested_dict()
materials

{'20PN1150F_251114_C67_test': {'youngs_modulus': 192500,
  'poissons_ratio': 0.3,
  'yield_stress': 370,
  'density': 7650}}

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Dict, Iterable, Optional


def safe_get_variable(mcad, key: str) -> Any:
    """Motor-CAD get_variable를 안전하게 호출 (없으면 None)."""
    try:
        return mcad.get_variable(key)
    except Exception:
        return None


@dataclass
class MechanicalProps:
    data: Dict[str, Any] = field(default_factory=dict)

    def update(self, props: Dict[str, Any], *, overwrite: bool = True, drop_none: bool = True) -> "MechanicalProps":
        for k, v in (props or {}).items():
            if drop_none and v is None:
                continue
            if overwrite or k not in self.data:
                self.data[k] = v
        return self


@dataclass
class ThermalProps:
    data: Dict[str, Any] = field(default_factory=dict)

    def update(self, props: Dict[str, Any], *, overwrite: bool = True, drop_none: bool = True) -> "ThermalProps":
        for k, v in (props or {}).items():
            if drop_none and v is None:
                continue
            if overwrite or k not in self.data:
                self.data[k] = v
        return self


@dataclass
class MagneticProps:
    data: Dict[str, Any] = field(default_factory=dict)

    def update(self, props: Dict[str, Any], *, overwrite: bool = True, drop_none: bool = True) -> "MagneticProps":
        for k, v in (props or {}).items():
            if drop_none and v is None:
                continue
            if overwrite or k not in self.data:
                self.data[k] = v
        return self


@dataclass
class MaterialInfo:
    name: str
    mechanical: MechanicalProps = field(default_factory=MechanicalProps)
    thermal: ThermalProps = field(default_factory=ThermalProps)
    magnetic: MagneticProps = field(default_factory=MagneticProps)

    def to_nested_dict(self) -> Dict[str, Dict[str, Any]]:
        return {
            "mechanical": dict(self.mechanical.data),
            "thermal": dict(self.thermal.data),
            "magnetic": dict(self.magnetic.data),
        }


class MaterialLibrary:
    def __init__(self) -> None:
        self._materials: Dict[str, MaterialInfo] = {}

    def get_or_create(self, name: Optional[str]) -> Optional[MaterialInfo]:
        if name is None:
            return None
        name = str(name).strip()
        if not name:
            return None
        mat = self._materials.get(name)
        if mat is None:
            mat = MaterialInfo(name=name)
            self._materials[name] = mat
        return mat

    def upsert_mechanical(self, name: Optional[str], props: Dict[str, Any] | None = None, *, overwrite: bool = True) -> Optional[MaterialInfo]:
        mat = self.get_or_create(name)
        if mat is None:
            return None
        mat.mechanical.update(props or {}, overwrite=overwrite)
        return mat

    def upsert_thermal(self, name: Optional[str], props: Dict[str, Any] | None = None, *, overwrite: bool = True) -> Optional[MaterialInfo]:
        mat = self.get_or_create(name)
        if mat is None:
            return None
        mat.thermal.update(props or {}, overwrite=overwrite)
        return mat

    def upsert_magnetic(self, name: Optional[str], props: Dict[str, Any] | None = None, *, overwrite: bool = True) -> Optional[MaterialInfo]:
        mat = self.get_or_create(name)
        if mat is None:
            return None
        mat.magnetic.update(props or {}, overwrite=overwrite)
        return mat

    def get(self, name: str) -> Optional[MaterialInfo]:
        return self._materials.get(name)

    def names(self) -> Iterable[str]:
        return self._materials.keys()

    def to_nested_dict(self) -> Dict[str, Dict[str, Dict[str, Any]]]:
        return {name: mat.to_nested_dict() for name, mat in self._materials.items()}

    def __getitem__(self, name: str) -> MaterialInfo:
        return self._materials[name]

    def __contains__(self, name: str) -> bool:
        return name in self._materials


lib = MaterialLibrary()  # 재료 라이브러리 (재료명 -> MaterialInfo)

In [ ]:
# syRe(Material Library .mat) 호환 export 유틸
# syRe는 MatList(cellstr) + MatLib(cell array of struct) 형태를 사용함.

from typing import Literal
import math
import numpy as np

SyreMaterialType = Literal["iron", "conductor", "layer", "sleeve"]

def _mpa_to_gpa(x: Any) -> Any:
    if x is None:
        return None
    try:
        x = float(x)
    except Exception:
        return x
    # Motor-CAD 값이 MPa(예: 192500)로 들어오면 GPa(192.5)로 변환
    return x / 1000.0 if x > 1000 else x

def material_to_syre_struct(mat: MaterialInfo, kind: SyreMaterialType) -> Dict[str, Any]:
    mech = mat.mechanical.data
    th = mat.thermal.data
    mag = mat.magnetic.data

    out: Dict[str, Any] = {"MatName": mat.name}

    if kind == "iron":
        # syRe iron fields: MatName, sigma_max[MPa], kgm3, alpha,beta,kh,ke, BH, E[GPa]
        out.update({
            "sigma_max": mech.get("yield_stress"),
            "kgm3": mech.get("density"),
            "E": _mpa_to_gpa(mech.get("youngs_modulus")),
            "alpha": mag.get("ironloss_alpha", 0),
            "beta": mag.get("ironloss_beta", 0),
            "kh": mag.get("ironloss_kh", 0),
            "ke": mag.get("ironloss_ke", 0),
        })
        # BH curve는 (H,B) 2열 형태를 기대하는 케이스가 많음. 없으면 생략.
        bh = mag.get("BH") or mag.get("bh")
        if bh is not None:
            out["BH"] = np.asarray(bh)
        return out

    if kind == "conductor":
        # syRe conductor fields: MatName, sigma[S/m], kgm3, alpha[1/°C]
        out.update({
            "sigma": th.get("electrical_conductivity", th.get("sigma")),
            "kgm3": mech.get("density", th.get("density")),
            "alpha": th.get("temp_coeff", th.get("alpha", 0.0)),
        })
        return out

    if kind == "sleeve":
        # syRe sleeve fields: MatName, sigma_max[MPa], kgm3, E[GPa]
        out.update({
            "sigma_max": mech.get("yield_stress"),
            "kgm3": mech.get("density"),
            "E": _mpa_to_gpa(mech.get("youngs_modulus")),
        })
        return out

    if kind == "layer":
        # syRe layer(자석) fields: MatName, mu, kgm3, Br, sigmaPM, temp{temp,Br,Bd}, Hc
        mu = mag.get("relative_permeability", mag.get("mu", 1.0))
        br = mag.get("remanence_T", mag.get("Br", 0.0))
        sigma_pm = th.get("electrical_conductivity", mag.get("sigmaPM", 0.0))
        out.update({
            "mu": mu,
            "kgm3": mech.get("density", th.get("density")),
            "Br": br,
            "sigmaPM": sigma_pm,
        })
        # 온도 의존 Br/Bd (없으면 기본 형태로 생성)
        temp_C = mag.get("temp_C", [20])
        br_vs_t = mag.get("Br_vs_temp", [0])
        bd_vs_t = mag.get("Bd_vs_temp", [0])
        out["temp"] = {
            "temp": np.asarray(temp_C),
            "Br": np.asarray(br_vs_t),
            "Bd": np.asarray(bd_vs_t),
        }
        # Hc = Br/(mu*(4e-7*pi))  (syRe add_material_layer.m 로직 동일)
        try:
            out["Hc"] = float(br) / (float(mu) * (4e-7 * math.pi))
        except Exception:
            pass
        return out

    raise ValueError(f"Unsupported kind: {kind}")

def export_syre_matlib(lib: MaterialLibrary, kind: SyreMaterialType, out_mat_path: str, *, names: Optional[Iterable[str]] = None) -> None:
    """syRe의 *.mat(MaterialLibrary) 형식으로 export. SciPy가 있으면 .mat 저장까지 수행."""
    if names is None:
        names = list(lib.names())
    MatList = list(names)
    MatLib = [material_to_syre_struct(lib[n], kind) for n in MatList]

    try:
        from scipy.io import savemat  # type: ignore
    except Exception as e:
        raise ImportError("scipy가 필요합니다. `pip install scipy` 후 다시 시도하세요.") from e

    savemat(out_mat_path, {
        "MatList": np.asarray(MatList, dtype=object),
        "MatLib": np.asarray(MatLib, dtype=object),
    }, do_compression=True)

    print(f"Saved syRe material library: {out_mat_path}")

In [ ]:
# 사용 예시 (syRe *.mat로 저장)
# 주의: 기존 custom_*.mat을 덮어쓸 수 있으니, 처음엔 별도 파일명으로 테스트 추천

# 예: 철심(iron) 재료만 export
# export_syre_matlib(lib, "iron", r"D:\\KangDH\\git_syRe\\materialLibrary\\custom_iron_from_motorcad.mat",
#                   names=[RotorMaterialName])

# 예: 자석(layer) 재료 export (자기/열 데이터 채운 뒤)
# export_syre_matlib(lib, "layer", r"D:\\KangDH\\git_syRe\\materialLibrary\\custom_layer_from_motorcad.mat",
#                   names=[MagnetMaterialName])

In [ ]:
# Magnet 재료도 라이브러리에 추가 (섹션 분리: mechanical/thermal/magnetic)
MagnetMaterialName = mcad.get_variable('Material_Magnet')

# 예시로 Mechanical 섹션에 기계물성 넣기 (Motor-CAD에 없으면 None으로 스킵됨)
lib.upsert_mechanical(MagnetMaterialName, {
    "youngs_modulus": safe_get_variable(mcad, "YoungsCoefficient_Magnet"),
    "poissons_ratio": safe_get_variable(mcad, "PoissonsRatio_Magnet"),
    "yield_stress": safe_get_variable(mcad, "YieldStress_Magnet"),
    "density": safe_get_variable(mcad, "Density_Magnet"),
})

# (원하면) 자기/열 물성도 이렇게 분리해서 추가 가능:
# lib.upsert_magnetic(MagnetMaterialName, {"relative_permeability": safe_get_variable(mcad, "RelativePermeability_Magnet")})
# lib.upsert_thermal(MagnetMaterialName, {"thermal_conductivity": safe_get_variable(mcad, "ThermalConductivity_Magnet")})

materials = lib.to_nested_dict()
materials

{'20PN1150F_251114_C67_test': {'youngs_modulus': 192500,
  'poissons_ratio': 0.3,
  'yield_stress': 370,
  'density': 7650},
 'NEOREC48DUH_251114_C67_test': {'youngs_modulus': 170000,
  'poissons_ratio': 0.27,
  'yield_stress': 0,
  'density': 7550}}